In [3]:
import pandas as pd 
import numpy as np


df = pd.read_csv('../data/raw/salary_benchmark.csv')


df_copy = df.copy()

df_copy = df_copy.drop(columns=['employee_id']).drop_duplicates()

In [5]:

#copy paste from 01_eda.ipynb
fix_cols = ['job_title', 'education_level', 'seniority_level']
for col in fix_cols:
    df[col] = df[col].str.strip().str.title()

value_map = {
    "Ml Engineer": "ML Engineer",
    "Hr Specialist": "HR Specialist",
    "Bachelor'S": "Bachelor",
    "Bachelors": "Bachelor",
    "Master'S": "Master",
    "Masters": "Master",
    "Mba": "MBA", 
    "Associate'S": "Associate",
    "Ph.D.": "PhD",
    "Phd": "PhD",
    "Doctorate": "PhD",
    "Mid-Level": "Mid"
}

for col in fix_cols:
    df[col] = df[col].replace(value_map)

## Impute

In [7]:
new_hire = df['years_at_company'] < 1 
df_copy.loc[new_hire & df['performance_rating'].isnull(), 'performance_rating'] = 3.0
df_copy['performance_rating'] = df_copy['performance_rating'].fillna(df_copy['performance_rating'].median())

df_copy['num_certifications'] = df_copy.groupby('job_title')['num_certifications'].transform(
    lambda x: x.fillna(x.median())
)

df_copy['years_experience'] = df_copy.groupby('seniority_level')['years_experience'].transform(
    lambda x: x.fillna(x.median())
)

print("Remaining Missing Values:", df_copy.isnull().sum().sum())

Remaining Missing Values: 0


In [11]:
invalid_experience = df_copy['years_experience'] > (df_copy['age'] - 18)
df_copy.loc[invalid_experience, 'years_experience'] = df_copy.loc[invalid_experience, 'age'] - 18

invalid_works = df_copy['years_at_company'] > df_copy['years_experience']
df_copy.loc[invalid_works, 'years_at_company'] = df_copy.loc[invalid_works, 'years_experience']

df_copy = df_copy[df_copy['base_salary'] >= 33000].copy()


print(f"Cleaned Dataset Shape: {df_copy.shape}")


Cleaned Dataset Shape: (5500, 16)
